In [1]:
from datetime import datetime

print(datetime.now())
# data preprocessing
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import collections
from collections import defaultdict
import os
import sys
import shutil
from collections import Counter

# the full input files pathes are here
DATA_PATH_stages = "../data/aki_preprocessing/kdigo_stages_measured.csv"
DATA_PATH_labs = "../data/aki_preprocessing/labs-kdigo_stages_measured.csv"
DATA_PATH_vitals = "../data/aki_preprocessing/vitals-kdigo_stages_measured.csv"
DATA_PATH_vents = "../data/aki_preprocessing/vents-vasopressor-sedatives-kdigo_stages_measured.csv"
DATA_PATH_detail = "../data/aki_preprocessing/icustay_detail-kdigo_stages_measured.csv"
SEPARATOR = ";"

2024-11-20 23:11:11.369147


In [2]:
# the output pathes are here
OUTPUT_PATH = "data/AKI_fts2"

In [3]:
# Set parameter as constant

# which classifier to use, only run one classifier at one time
ALL_STAGES = False  # not binary label, each class separately 0,1,2,3

CLASS1 = True  # AnyAKI
CLASS2 = False  # ModerateSevereAKI
CLASS3 = False  # SevereAKI


MAX_FEATURE_SET = True

# resampling  and imputing
TIME_SAMPLING = True
SAMPLING_INTERVAL = "1H"
# RESAMPLE_LIMIT = 16 # 4 days*6h interval

# if MOST_COMMON is not applied,sampling with different strategies per kind of variable,
# numeric variables use mean value, categorical variables use max value
MOST_COMMON = False  # resampling with most common

# fit Yereva's time span
MAX_HOUR = 48

# How much time the prediction should occur (hours)
HOURS_AHEAD = 48

IMPUTE_EACH_ID = True  # imputation within each icustay_id with most common value | False like in the original notebook
IMPUTE_COLUMN = False  # imputation based on whole column
IMPUTE_METHOD = "most_frequent"
FILL_VALUE = 0  # fill missing value and ragged part of 3d array

# Age constraints: adults
ADULTS_MIN_AGE = 18
ADULTS_MAX_AGE = -1

NORMALIZATION = "min-max"
NORM_TYPE = "min_max"

CAPPING = True
if CAPPING:
    CAPPING_THRESHOLD_UPPER = 0.99
    CAPPING_THRESHOLD_LOWER = 0.01


# use random split or fixed train/val/test set
RANDOM_SPLIT = True
FIXED = False
RANDOM_SEED = 42
SPLIT_SIZE = 0.2

# set changable info corresponding to each classifier as variables

min_set = ["icustay_id", "charttime", "creat", "uo_rt_6hr", "uo_rt_12hr", "uo_rt_24hr", "aki_stage"]

max_set = [
    "icustay_id",
    "charttime",
    "aki_stage",
    "hadm_id",
    #"albumin_avg", # Not in original notebook
    "aniongap_avg",
    "bicarbonate_avg",
    "bilirubin_avg",
    #"bun_avg", # Not in original notebook
    "chloride_avg",
    "creat",
    "diasbp_mean",
    "glucose_avg",
    "heartrate_mean",
    "hematocrit_avg",
    "hemoglobin_avg",
    "potassium_avg",
    "resprate_mean",
    "sodium_avg",
    "spo2_mean",
    "sysbp_mean",
    "uo_rt_12hr",
    "uo_rt_24hr",
    "uo_rt_6hr",
    "wbc_avg",
    "sedative",
    "vasopressor",
    "vent",
    "age",
    "F",
    "M",
    "asian",
    "black",
    "hispanic",
    "native",
    "other",
    "unknown",
    "white",
    "ELECTIVE",
    "EMERGENCY",
    "URGENT",
]
print(f'Max set: {len(max_set)}')

Max set: 39


In [4]:
# Some functions used later
def cap_data(df):
    print("Capping between the {} and {} quantile".format(CAPPING_THRESHOLD_LOWER, CAPPING_THRESHOLD_UPPER))

    # Original line ['icustay_id', 'charttime', 'aki_stage']
    cap_mask = df.columns.difference(['icustay_id', 'charttime', 'aki_stage'])

    # cap_mask = df.columns.difference(["icustay_id", "charttime", "aki_stage", "subject_id", "intime", "HOURS"])

    print("cap_maks", cap_mask)

    # Filtrar solo columnas numéricas
    numeric_cols = df[cap_mask].select_dtypes(include=[np.number]).columns
    print("numeric_cols", numeric_cols)

    df[numeric_cols] = df[numeric_cols].clip(
        df[numeric_cols].quantile(CAPPING_THRESHOLD_LOWER),
        df[numeric_cols].quantile(CAPPING_THRESHOLD_UPPER),
        axis=1
    )

    return df


def normalise_data(df, norm_mask):
    print("Normalizing in [0,1] with {} normalization".format(NORMALIZATION))

    df[norm_mask] = (df[norm_mask] - df[norm_mask].min()) / (df[norm_mask].max() - df[norm_mask].min())

    return df


# impute missing value in resampleing data with most common based on each id
def fast_mode(df, key_cols, value_col):
    """Calculate a column mode, by group, ignoring null values.

    key_cols : list of str - Columns to groupby for calculation of mode.
    value_col : str - Column for which to calculate the mode.

    Return
    pandas.DataFrame
        One row for the mode of value_col per key_cols group. If ties, returns the one which is sorted first."""
    return (
        df.groupby(key_cols + [value_col])
        .size()
        .to_frame("counts")
        .reset_index()
        .sort_values("counts", ascending=False)
        .drop_duplicates(subset=key_cols)
    ).drop("counts", axis=1)


# get max shape of 3d array
def get_dimensions(array, level=0):
    yield level, len(array)
    try:
        for row in array:
            yield from get_dimensions(row, level + 1)
    except TypeError:  # not an iterable
        pass


def get_max_shape(array):
    dimensions = defaultdict(int)
    for level, length in get_dimensions(array):
        dimensions[level] = max(dimensions[level], length)
    return [value for _, value in sorted(dimensions.items())]


# pad the ragged 3d array to rectangular shape based on max size
def iterate_nested_array(array, index=()):
    try:
        for idx, row in enumerate(array):
            yield from iterate_nested_array(row, (*index, idx))
    except TypeError:  # final level
        yield (*index, slice(len(array))), array  # think of the types


def pad(array, fill_value):
    dimensions = get_max_shape(array)
    result = np.full(dimensions, fill_value, dtype=np.float64)
    for index, value in iterate_nested_array(array):
        result[index] = value
    return result

# Read csv files

## kdigo_stages_measured

In [5]:
print("read csv files")

# reading csv files
X = pd.read_csv(DATA_PATH_stages, sep=SEPARATOR)
X.drop(["aki_stage_creat", "aki_stage_uo"], axis=1, inplace=True)

# remove totally empty rows
X = X.dropna(how="all", subset=["creat", "uo_rt_6hr", "uo_rt_12hr", "uo_rt_24hr", "aki_stage"])
print("convert charttime to timestamp")
X["charttime"] = pd.to_datetime(X["charttime"])

# merge rows if they have exact timestamp within same icustay_id AL : it substitutes missing values with zero
# X = X.groupby(['icustay_id', 'charttime']).sum().reset_index(['icustay_id', 'charttime'])

read csv files
convert charttime to timestamp


In [6]:
unique_subjects = X['subject_id'].nunique()
print(f"Unique subject_id count: {unique_subjects}")

Unique subject_id count: 50878


In [7]:
X.head()

,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,creat,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed
0,10000032,29079034,39553978,2180-07-23 06:39:00,NaN,NaN,0.7,NaN,NaN,NaN,NaN,0,0
1,10000032,29079034,39553978,2180-07-23 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,10000032,29079034,39553978,2180-07-23 21:45:00,0.7,0.7,0.5,NaN,NaN,NaN,NaN,0,0
3,10000980,26913865,39765666,2189-06-27 06:48:00,NaN,NaN,2.3,NaN,NaN,NaN,NaN,0,0
4,10000980,26913865,39765666,2189-06-27 09:08:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [8]:
X['stay_id'].nunique()

73092

In [9]:
patients_per_aki_stage = X.groupby('aki_stage')['subject_id'].nunique()
print(patients_per_aki_stage)

aki_stage
0    50878
1    33282
2    24507
3     8891
Name: subject_id, dtype: int64


In [10]:
import pandas as pd

# Define the paths to the CSV files
file_paths = [
    "./data/AKI_fts2/test_listfile.csv",
    "./data/AKI_fts2/train_listfile.csv",
    "./data/AKI_fts2/val_listfile.csv"
]

# Read the CSV files into dataframes
dfs = [pd.read_csv(file_path) for file_path in file_paths]

# Concatenate the dataframes
combined_df = pd.concat(dfs, ignore_index=True)

# Calculate the total number of records
total_records = combined_df.shape[0]

# Calculate the number of records with y_true equal to 1
y_true_1_count = combined_df[combined_df['y_true'] == 1].shape[0]

print(f"Total records: {total_records}")
print(f"Records with y_true == 1: {y_true_1_count}")

# Calculate the number of records and y_true == 1 for each file
for file_path, df in zip(file_paths, dfs):
    file_name = file_path.split('/')[-1]
    record_count = df.shape[0]
    y_true_1_count = df[df['y_true'] == 1].shape[0]
    y_true_proportion = df['y_true'].value_counts(normalize=True)
    print(f"File: {file_name}")
    print(f"  Total records: {record_count}")
    print(f"  Records with y_true == 1: {y_true_1_count}")
    print(f"  Proportion of y_true values:\n{y_true_proportion}")

Total records: 65382
Records with y_true == 1: 45024
File: test_listfile.csv
  Total records: 6368
  Records with y_true == 1: 4384
  Proportion of y_true values:
y_true
1    0.688442
0    0.311558
Name: proportion, dtype: float64
File: train_listfile.csv
  Total records: 51996
  Records with y_true == 1: 35809
  Proportion of y_true values:
y_true
1    0.688688
0    0.311312
Name: proportion, dtype: float64
File: val_listfile.csv
  Total records: 7018
  Records with y_true == 1: 4831
  Proportion of y_true values:
y_true
1    0.688373
0    0.311627
Name: proportion, dtype: float64


## icustay_detail-kdigo_stages_measured

In [11]:
dataset_detail = pd.read_csv(DATA_PATH_detail, sep=SEPARATOR)  # age constraint
# keep "intime" to calculate Hours in Yereva

# subject_id;hadm_id;stay_id;gender;anchor_age;anchor_year;anchor_year_group;admittime;dischtime;deathtime;
# race -> ethnicity
# deathtime -> dod

dataset_detail.drop(
    [
        "dod",
        "admittime",
        "dischtime",
        "los_hospital",
        'ethnicity', # In original notebook
        #"race", # Not in original notebook
        "hospital_expire_flag",
        "hospstay_seq",
        "first_hosp_stay",
        #"intime", # In original notebook
        "outtime",
        "los_icu",
        "icustay_seq",
        "first_icu_stay",
    ],
    axis=1,
    inplace=True,
    errors="ignore",
)

In [12]:
dataset_detail.head()

,subject_id,hadm_id,stay_id,gender,admission_age,race,icu_intime,icu_outtime,subject_id.1,gender.1,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race.1,edregtime,edouttime,hospital_expire_flag.1
0,10000032,29079034,39553978,F,52.559969,WHITE,2180-07-23 14:00:00,2180-07-23 23:50:47,10000032,F,...,P30KEH,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
1,10000980,26913865,39765666,F,76.486231,BLACK/AFRICAN AMERICAN,2189-06-27 08:42:00,2189-06-27 20:38:27,10000980,F,...,P30KEH,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,2189-06-27 06:25:00,2189-06-27 08:42:00,0
2,10001217,24597018,37067082,F,55.881486,WHITE,2157-11-20 19:18:02,2157-11-21 22:08:00,10001217,F,...,P4645A,EMERGENCY ROOM,HOME HEALTH CARE,Other,?,MARRIED,WHITE,2157-11-18 17:38:00,2157-11-19 01:24:00,0
3,10001217,27703517,34592300,F,55.962942,WHITE,2157-12-19 15:42:24,2157-12-20 14:27:41,10001217,F,...,P99698,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,?,MARRIED,WHITE,NaN,NaN,0
4,10001725,25563031,31205490,F,46.275517,WHITE,2110-04-11 15:52:22,2110-04-12 23:59:56,10001725,F,...,P35SU0,PACU,HOME,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,0


In [13]:
print(f"Unique subject_id count: {dataset_detail['subject_id'].nunique()}")

Unique subject_id count: 50878


In [14]:
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'subject_id.1', 'gender.1', 'anchor_age',
       'anchor_year', 'anchor_year_group', 'dod.1', 'subject_id.2',
       'hadm_id.1', 'admittime.1', 'dischtime.1', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status',
       'race.1', 'edregtime', 'edouttime', 'hospital_expire_flag.1'],
      dtype='object')

Vars duplicated
* subject_id, subject_id.1, subject_id.2
* hadm_id, hadm_id.1
* gender, gender.1
* dod (Previously removed), dod.1
* admittime(Previously removed), admittime.1
* dischtime(Prev removed), dischtime.1
* race, race.1
* hospital_expire_flag(Prev removed), hospital_expire_flag.1

In [15]:
inconsistent_rows = dataset_detail[['race', 'race.1']].apply(lambda row: row.nunique() != 1, axis=1)
if inconsistent_rows.any():
    print("There are inconsistent rows in the dataset.")
else:
    print("All rows are consistent.")

All rows are consistent.


In [16]:
# Remove duplicated columns, and the ones that are copy of previous removed columns
dataset_detail.drop(
    ['subject_id.1','subject_id.2','hadm_id.1','gender.1','dod.1','admittime.1','dischtime.1','race.1','hospital_expire_flag.1'],
    axis=1,
    inplace=True,
    errors="ignore",
)
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime'],
      dtype='object')

In [17]:
print("convert intime to timestamp")
dataset_detail["intime"] = pd.to_datetime(dataset_detail["icu_intime"])

INTIME = pd.DataFrame()
INTIME["stay_id"] = dataset_detail["stay_id"]
INTIME["intime"] = dataset_detail["intime"]

convert intime to timestamp


## labs-kdigo_stages_measured

In [18]:
dataset_labs = pd.read_csv(DATA_PATH_labs, sep=SEPARATOR)  # 'bands lactate platelet ptt inr pt
dataset_labs.drop(
    [
        "albumin_min",
        "albumin_max",
        "bilirubin_min",
        "bilirubin_max",
        "bands_min",
        "bands_max",
        "lactate_min",
        "lactate_max",
        "platelet_min",
        "platelet_max",
        "ptt_min",
        "ptt_max",
        "inr_min",
        "inr_max",
        "pt_min",
        "pt_max",
    ],
    axis=1,
    inplace=True,
)
dataset_labs.head()

,subject_id,hadm_id,stay_id,charttime,aniongap_min,aniongap_max,bicarbonate_min,bicarbonate_max,creatinine_min,creatinine_max,...,hemoglobin_min,hemoglobin_max,potassium_min,potassium_max,sodium_min,sodium_max,bun_min,bun_max,wbc_min,wbc_max
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,12.5,12.5,4.4,4.4,141.0,141.0,NaN,NaN,NaN,NaN
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,NaN,NaN,NaN,NaN,...,10.9,10.9,4.2,4.2,142.0,142.0,NaN,NaN,NaN,NaN
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,12.0,19.0,19.0,0.9,0.9,...,10.8,10.8,4.4,4.4,142.0,142.0,22.0,22.0,17.0,17.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
dataset_labs = dataset_labs.dropna(subset=["charttime"])
dataset_labs = dataset_labs.dropna(subset=dataset_labs.columns[4:], how="all")
dataset_labs["charttime"] = pd.to_datetime(dataset_labs["charttime"])
dataset_labs = dataset_labs.sort_values(by=["stay_id", "charttime"])

In [20]:
dataset_labs.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'aniongap_min',
       'aniongap_max', 'bicarbonate_min', 'bicarbonate_max', 'creatinine_min',
       'creatinine_max', 'chloride_min', 'chloride_max', 'glucose_min',
       'glucose_max', 'hematocrit_min', 'hematocrit_max', 'hemoglobin_min',
       'hemoglobin_max', 'potassium_min', 'potassium_max', 'sodium_min',
       'sodium_max', 'bun_min', 'bun_max', 'wbc_min', 'wbc_max'],
      dtype='object')

In [21]:
dataset_labs['subject_id'].nunique()

50399

## vitals-kdigo_stages_measured & vents-vasopressor-sedatives-kdigo_stages_measured

In [22]:
if MAX_FEATURE_SET:
    dataset_vitals = pd.read_csv(DATA_PATH_vitals, sep=SEPARATOR)
    dataset_vents = pd.read_csv(DATA_PATH_vents, sep=SEPARATOR)
    # dataset_icd = pd.read_csv(DATA_PATH_icd, sep= SEPARATOR)
    dataset_vitals.drop(
        [
            "heartrate_min",
            "heartrate_max",
            "sysbp_min",
            "sysbp_max",
            "diasbp_min",
            "diasbp_max",
            "meanbp_min",
            "meanbp_max",
            "meanbp_mean",
            "tempc_min",
            "tempc_max",
            "tempc_mean",
            "resprate_min",
            "resprate_max",
            "spo2_min",
            "spo2_max",
            "glucose_min",
            "glucose_max",
        ],
        axis=1,
        inplace=True,
    )
    print("convert charttime to timestamp")
    dataset_vitals["charttime"] = pd.to_datetime(dataset_vitals["charttime"])
    dataset_vents["charttime"] = pd.to_datetime(dataset_vents["charttime"])
    dataset_vitals = dataset_vitals.sort_values(by=["stay_id", "charttime"])
    dataset_vents = dataset_vents.sort_values(by=["stay_id", "charttime"])
    # AL drop those where all columns are nan (empty rows)
    dataset_vitals = dataset_vitals.dropna(subset=dataset_vitals.columns[4:], how="all")

convert charttime to timestamp


In [23]:
dataset_vitals.head()

,subject_id,hadm_id,stay_id,charttime,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,glucose_mean
2391773,12466550,23998182,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,14.0,NaN,NaN
2391774,12466550,23998182,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,100.0,NaN
2391775,12466550,23998182,30000153,2174-09-29 12:06:00,100.0,136.0,74.0,NaN,NaN,NaN
2391777,12466550,23998182,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,18.0,NaN,NaN
2391778,12466550,23998182,30000153,2174-09-29 13:00:00,104.0,113.0,77.0,16.0,100.0,NaN


In [24]:
dataset_vitals['subject_id'].nunique()

50878

In [25]:
dataset_vents.head()

,stay_id,charttime,vent,vasopressor,sedative
0,30000153,2174-09-29 10:16:00,0,0,0
1,30000153,2174-09-29 12:12:00,1,0,1
2,30000153,2174-09-29 12:27:00,1,0,1
3,30000153,2174-09-29 13:27:00,1,0,1
4,30000153,2174-09-29 14:00:00,1,0,0


In [26]:
# Why this?
# def break_up_stays_by_subject(stays, output_path, subjects=None, verbose=1):
#     subjects = stays.subject_id.unique() if subjects is None else subjects
#     nb_subjects = subjects.shape[0]
#     for i, subject_id in enumerate(subjects):
#         if verbose:
#             sys.stdout.write("\rSUBJECT {0} of {1}...".format(i + 1, nb_subjects))
#         dn = os.path.join(output_path, str(subject_id))
#         try:
#             os.makedirs(dn)
#         except:
#             pass

#         stays.ix[stays.subject_id == subject_id].sort_values(by="intime").to_csv(
#             os.path.join(dn, "stays.csv"), index=False
#         )
#     if verbose:
#         sys.stdout.write("DONE!\n")

# Calculate avg of Lab features

In [27]:
print("compute avg from min/max in labs file")
print(datetime.now())
# Labs file: instead of min and max their avg
counter = 0
col1 = 4
col2 = 5
null_l = []  # no null values in those that are different
changed = 0  # 4316 records changed to avg

# 11 pairs of columns to be averaged:
# 'aniongap_min','aniongap_max', 
# 'bicarbonate_min', 'bicarbonate_max',
# 'creatinine_min','creatinine_max',
# 'chloride_min', 'chloride_max',
# 'glucose_min', 'glucose_max',
# 'hematocrit_min', 'hematocrit_max',
# 'hemoglobin_min', 'hemoglobin_max',
# 'potassium_min', 'potassium_max',
# 'sodium_min', 'sodium_max',
# 'bun_min', 'bun_max',
# 'wbc_min', 'wbc_max'

while counter < 11:
    row = 0
    # find where min and max are different and save their row indices
    # Calculate avg between min and max. Save it in min column, remove the max column,
    # and continue with the next pair of columns.
    while row < len(dataset_labs):
        a = dataset_labs.iloc[row, col1]
        b = dataset_labs.iloc[row, col2]
        if a == b or (np.isnan(a) and np.isnan(b)):
            pass
        elif a != b:
            changed += 1
            avg = (a + b) / 2
            dataset_labs.iloc[row, col1] = avg
            if (np.isnan(a) and ~np.isnan(b)) or (np.isnan(b) and ~np.isnan(a)):
                null_l.append(row)
        else:
            print(a)
            print(b)
        row += 1
    # delete the redundant column max, update counters
    dataset_labs.drop(dataset_labs.columns[col2], axis=1, inplace=True)
    counter = counter + 1
    col1 = col1 + 1
    col2 = col2 + 1

dataset_labs.columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "charttime",
    "aniongap_avg",
    "bicarbonate_avg",
    "creatinine_avg",
    "chloride_avg",
    "glucose_avg",
    "hematocrit_avg",
    "hemoglobin_avg",
    "potassium_avg",
    "sodium_avg",
    "bun_avg",  # Blood Urea Nitrogen
    "wbc_avg",  # White blood cells
]
if len(null_l) > 0:
    print("null values encountered ", len(null_l))
print(datetime.now())

compute avg from min/max in labs file
2024-11-19 23:00:40.168244
2024-11-19 23:18:14.555081


In [28]:
len(null_l)

0

In [29]:
dataset_labs.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'aniongap_avg',
       'bicarbonate_avg', 'creatinine_avg', 'chloride_avg', 'glucose_avg',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'sodium_avg',
       'bun_avg', 'wbc_avg'],
      dtype='object')

# Merge creatinine 

In [30]:
print("Merge creatinine and glucose.")
# merge creatinine from labs and set with labels
creat_l = dataset_labs[["stay_id", "charttime", "creatinine_avg"]].copy()
creat_l = creat_l.dropna(subset=["creatinine_avg"])

creat = X[["stay_id", "charttime", "creat"]].copy()
creat = creat.dropna(subset=["creat"])

creat_l = creat_l.rename(columns={"creatinine_avg": "creat"})
# creat = creat.append(creat_l, ignore_index=True) # for old version of pandas
creat = pd.concat([creat, creat_l], ignore_index=True)
creat.drop_duplicates(inplace=True)
creat.head()

Merge creatinine and glucose.


,stay_id,charttime,creat
0,39553978,2180-07-23 06:39:00,0.7
1,39553978,2180-07-23 21:45:00,0.5
2,39765666,2189-06-27 06:48:00,2.3
3,37067082,2157-11-18 18:30:00,0.6
4,37067082,2157-11-20 08:14:00,0.7


In [31]:
dataset_labs.head()

,subject_id,hadm_id,stay_id,charttime,aniongap_avg,bicarbonate_avg,creatinine_avg,chloride_avg,glucose_avg,hematocrit_avg,hemoglobin_avg,potassium_avg,sodium_avg,bun_avg,wbc_avg
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,NaN,110.0,158.0,38.0,12.5,4.4,141.0,NaN,NaN
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,NaN,112.0,176.0,33.0,10.9,4.2,142.0,NaN,NaN
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,19.0,0.9,115.0,192.0,31.7,10.8,4.4,142.0,22.0,17.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,NaN,175.0,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
# delete old columns
dataset_labs.drop(["creatinine_avg"], axis=1, inplace=True)
dataset_labs = dataset_labs.dropna(subset=dataset_labs.columns[4:], how="all")
dataset_labs.head()

,subject_id,hadm_id,stay_id,charttime,aniongap_avg,bicarbonate_avg,chloride_avg,glucose_avg,hematocrit_avg,hemoglobin_avg,potassium_avg,sodium_avg,bun_avg,wbc_avg
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,110.0,158.0,38.0,12.5,4.4,141.0,NaN,NaN
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,112.0,176.0,33.0,10.9,4.2,142.0,NaN,NaN
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,19.0,115.0,192.0,31.7,10.8,4.4,142.0,22.0,17.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,175.0,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
X.head()

,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,creat,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed
0,10000032,29079034,39553978,2180-07-23 06:39:00,NaN,NaN,0.7,NaN,NaN,NaN,NaN,0,0
1,10000032,29079034,39553978,2180-07-23 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,10000032,29079034,39553978,2180-07-23 21:45:00,0.7,0.7,0.5,NaN,NaN,NaN,NaN,0,0
3,10000980,26913865,39765666,2189-06-27 06:48:00,NaN,NaN,2.3,NaN,NaN,NaN,NaN,0,0
4,10000980,26913865,39765666,2189-06-27 09:08:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [34]:
# X[X['subject_id']==10000980]

In [35]:
X.drop(["creat"], axis=1, inplace=True)
# merge new column
X = pd.merge(X, creat, on=["stay_id", "charttime"], sort=True, how="outer", copy=False)
X.head()

,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed,creat
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.2
1,12466550.0,23998182.0,30000153,2174-09-29 12:12:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,12466550.0,23998182.0,30000153,2174-09-29 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,12466550.0,23998182.0,30000153,2174-09-29 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
4,12466550.0,23998182.0,30000153,2174-09-29 15:37:00,1.2,1.2,NaN,NaN,NaN,NaN,0.0,0.0,0.9


# Merge glucose

Merge glucose from vitals and labs

In [36]:
dataset_vitals.head()

,subject_id,hadm_id,stay_id,charttime,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,glucose_mean
2391773,12466550,23998182,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,14.0,NaN,NaN
2391774,12466550,23998182,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,100.0,NaN
2391775,12466550,23998182,30000153,2174-09-29 12:06:00,100.0,136.0,74.0,NaN,NaN,NaN
2391777,12466550,23998182,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,18.0,NaN,NaN
2391778,12466550,23998182,30000153,2174-09-29 13:00:00,104.0,113.0,77.0,16.0,100.0,NaN


In [37]:
if MAX_FEATURE_SET:
    glucose_v = dataset_vitals[["subject_id", "hadm_id", "stay_id", "charttime", "glucose_mean"]].copy()
    glucose_v = glucose_v.dropna(subset=["glucose_mean"])
    glucose = dataset_labs[["subject_id", "hadm_id", "stay_id", "charttime", "glucose_avg"]].copy()
    glucose = glucose.dropna(subset=["glucose_avg"])
    glucose_v = glucose_v.rename(columns={"glucose_mean": "glucose_avg"})

    # glucose = glucose.append(glucose_v, ignore_index=True) # for old version of pandas
    glucose = pd.concat([glucose, glucose_v], ignore_index=True)

    glucose.drop_duplicates(inplace=True)
    
    # delete old columns
    dataset_labs.drop(["glucose_avg"], axis=1, inplace=True)
    dataset_vitals.drop(["glucose_mean"], axis=1, inplace=True)
    dataset_vitals = dataset_vitals.dropna(subset=dataset_vitals.columns[4:], how="all")
    
    # merge new column
    dataset_labs = pd.merge(
        dataset_labs,
        glucose,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "charttime",
        ],
        sort=True,
        how="outer",
        copy=False,
    )

dataset_labs = dataset_labs.sort_values(by=["stay_id", "charttime"], ignore_index=True)
X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)

In [38]:
dataset_labs.head()

,subject_id,hadm_id,stay_id,charttime,aniongap_avg,bicarbonate_avg,chloride_avg,hematocrit_avg,hemoglobin_avg,potassium_avg,sodium_avg,bun_avg,wbc_avg,glucose_avg
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,110.0,38.0,12.5,4.4,141.0,NaN,NaN,158.0
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,112.0,33.0,10.9,4.2,142.0,NaN,NaN,176.0
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,19.0,115.0,31.7,10.8,4.4,142.0,22.0,17.0,192.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,175.0


In [39]:
X = pd.merge(X, dataset_labs, on=["stay_id", "charttime"], how="outer", copy=False)

In [40]:
X.columns

Index(['subject_id_x', 'hadm_id_x', 'stay_id', 'charttime',
       'creat_low_past_7day', 'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr',
       'uo_rt_24hr', 'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed',
       'creat', 'subject_id_y', 'hadm_id_y', 'aniongap_avg', 'bicarbonate_avg',
       'chloride_avg', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'sodium_avg', 'bun_avg', 'wbc_avg', 'glucose_avg'],
      dtype='object')

In [41]:
# X[['subject_id_x', 'hadm_id_x','subject_id_y', 'hadm_id_y']].head(50)

# Initialize an array to store the indices of rows where subject_id_x and subject_id_y differ
differing_indices = []

# Iterate over the rows of the DataFrame
for index, row in X.iterrows():
    if pd.isna(row['subject_id_x']) and not pd.isna(row['subject_id_y']):
        X.at[index, 'subject_id_x'] = row['subject_id_y']
    elif not pd.isna(row['subject_id_x']) and not pd.isna(row['subject_id_y']):
        if row['subject_id_x'] != row['subject_id_y']:
            differing_indices.append(index)

# Print the indices of rows where subject_id_x and subject_id_y differ
print("Indices with differing subject_id values:", differing_indices)
# Initialize an array to store the indices of rows where hadm_id_x and hadm_id_y differ
differing_hadm_indices = []

# Iterate over the rows of the DataFrame
for index, row in X.iterrows():
    if pd.isna(row['hadm_id_x']) and not pd.isna(row['hadm_id_y']):
        X.at[index, 'hadm_id_x'] = row['hadm_id_y']
    elif not pd.isna(row['hadm_id_x']) and not pd.isna(row['hadm_id_y']):
        if row['hadm_id_x'] != row['hadm_id_y']:
            differing_hadm_indices.append(index)

# Print the indices of rows where hadm_id_x and hadm_id_y differ
print("Indices with differing hadm_id values:", differing_hadm_indices)


Indices with differing subject_id values: []
Indices with differing hadm_id values: []


In [42]:
# Drop the columns subject_id_y and hadm_id_y
X.drop(columns=['subject_id_y', 'hadm_id_y'], inplace=True)

# Rename the columns hadm_id_x and subject_id_x by removing the '_x'
X.rename(columns={'hadm_id_x': 'hadm_id', 'subject_id_x': 'subject_id'}, inplace=True)

# Display the updated columns
X.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg'],
      dtype='object')

# Merge vital signs

In [43]:
X = pd.merge(X, dataset_vitals, on=["stay_id", "charttime", "subject_id", "hadm_id"], how="outer", copy=False)

In [44]:
X.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg', 'heartrate_mean', 'sysbp_mean', 'diasbp_mean',
       'resprate_mean', 'spo2_mean'],
      dtype='object')

In [45]:
X = pd.merge(X, dataset_vents, on=["stay_id", "charttime"], how="outer", copy=False)

In [136]:
# X.drop(["subject_id"], axis = 1, inplace = True)

In [ ]:
# The following method was executed in the previous cell separately.

# print("Merging labs, vitals and vents files")
# if MAX_FEATURE_SET:
#     X = pd.merge(X, dataset_labs, on=["stay_id", "charttime"], how="outer", copy=False)
#     X = pd.merge(X, dataset_vitals, on=["stay_id", "charttime", "subject_id", "hadm_id"], how="outer", copy=False)
#     X = pd.merge(X, dataset_vents, on=["stay_id", "charttime"], how="outer", copy=False)
    # X.drop(["subject_id"], axis = 1, inplace = True)

# Removing patienes under the min age

In [ ]:
print("start preprocessing time dependent data")
dataset_detail = dataset_detail.loc[dataset_detail["anchor_age"] >= ADULTS_MIN_AGE] #admission_age
adults_icustay_id_list = dataset_detail["stay_id"].unique()
X = X[X.stay_id.isin(adults_icustay_id_list)].sort_values(by=["stay_id"], ignore_index=True)
X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)
adults_icustay_id_list = np.sort(adults_icustay_id_list)

start preprocessing time dependent data


In [47]:
print(f"Number of unique icustay_id: {len(adults_icustay_id_list)}")

Number of unique icustay_id: 73092


In [48]:
X_copy_2 = X.copy()
X.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg', 'heartrate_mean', 'sysbp_mean', 'diasbp_mean',
       'resprate_mean', 'spo2_mean', 'vent', 'vasopressor', 'sedative'],
      dtype='object')

In [ ]:
#Retrieve data
# X = X_copy_2.copy()

# Remove stay_id with less than 48 hrs

In [ ]:
print("drop stay_id with time span less than 48hrs")

def more_than_HOURS_ahead(adults_icustay_id_list, X):
    drop_list = []
    los_list = []  # calculating LOS (Length of Stay) ICU based on charttime
    long_stays_id = []  # LOS longer than MAX DAYS days
    last_charttime_list = []

    # Sian modified to above code, AL: seq_length = X.groupby(['stay_id'],as_index=False).size().to_frame('size')
    seq_length = X.groupby(["stay_id"], as_index=False).size()
    
    id_count = 0
    first_row_index = 0

    while id_count < len(adults_icustay_id_list):
        stay_id = adults_icustay_id_list[id_count]
        last_row_index = (
            first_row_index + seq_length.iloc[id_count, 1] - 1
        )  # Sian modified, AL: seq_length.iloc[id_count,0]-1
        first_time = X.iat[first_row_index, X.columns.get_loc("charttime")]
        last_time = X.iat[last_row_index, X.columns.get_loc("charttime")]
        los = round(float((last_time - first_time).total_seconds() / 60 / 60 / 24), 4)  # in days
        if los < 48 / 24:
            drop_list.append(stay_id)
        else:
            los_list.append(los)
            if los > 35:
                long_stays_id.append(stay_id)
                last_charttime_list.append(last_time)
        # udpate for the next stay_id
        first_row_index = last_row_index + 1
        id_count += 1

    if len(long_stays_id) != len(last_charttime_list):
        print("ERROR")
        
    print("%d long stays" % len(long_stays_id))
    # drop all the rows with the saved stay_id
    print("there are %d id-s shorter than 48 hours" % len(drop_list))
    X = X[~X.stay_id.isin(drop_list)]
    id_list = X["stay_id"].unique()
    X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)

    return id_list, X, long_stays_id, last_charttime_list

# id_list: list of unique icustay_id
id_list, X, long_stays_id, last_charttime_list = more_than_HOURS_ahead(adults_icustay_id_list, X)

long = pd.DataFrame()
long["stay_id"] = long_stays_id
long["last_time"] = last_charttime_list

drop stay_id with time span less than 48hrs
3183 long stays
there are 7679 id-s shorter than 48 hours


In [54]:
print(f"Unique stay_if {len(id_list)}")

unique_patients = X['subject_id'].nunique()
print(f"Unique patients in X: {unique_patients}")

Unique stay_if 65413
Unique patients in X: 45233


In [55]:
# Group by subject_id and check if any aki_stage is greater than 0
stayid_with_aki = X.groupby('stay_id')['aki_stage'].max().reset_index()
stayid_with_aki['has_aki'] = stayid_with_aki['aki_stage'] > 0

# Count the number of patients with and without AKI
aki_counts = stayid_with_aki['has_aki'].value_counts()
print(aki_counts)

has_aki
True     44654
False    20759
Name: count, dtype: int64


In [160]:
# deleting rows that are not within MAX_DAYS (35) period
# i = 0 # long df index
# drop_long_time = []
    
# while i < len(long_stays_id):
#     j = 0
#     all_rows = X.index[X['stay_id'] == long.loc[i,'stay_id']].tolist()
#     while j < len(all_rows):
#         time = X.iat[all_rows[j], X.columns.get_loc('charttime')]
#         # if keep last MAX_DAYS 
#         if (long.loc[i,'last_time'] - time).total_seconds() > 35*24*60*60:
#             drop_long_time.append(all_rows[j])
#             j +=1
#         else:
#             break
#     i +=1       
# X.drop(X.index[drop_long_time], inplace=True) 

# # checking for 48h min length again
# id_list, X, long_stays_id,last_charttime_list  = more_than_HOURS_ahead(id_list, X)
# dataset_detail = dataset_detail[dataset_detail.stay_id.isin(id_list)].sort_values(by=['stay_id'], ignore_index = True)

0 long stays
there are 4 id-s shorter than 48 hours


# Extract Label 

In [56]:
ALL_STAGES

False

In [57]:
print("binarise labels")
if ALL_STAGES:
    pass
elif CLASS1:
    # No AKI = 0, AKI = 1
    X.loc[X['aki_stage'] > 1, 'aki_stage'] = 1
elif CLASS2:
    X.loc[X['aki_stage'] < 2, 'aki_stage'] = 0
    X.loc[X['aki_stage'] > 1, 'aki_stage'] = 1
elif CLASS3:
    X.loc[X['aki_stage'] < 3, 'aki_stage'] = 0
    X.loc[X['aki_stage'] > 2, 'aki_stage'] = 1

binarise labels


In [58]:
# If any of the records presents aki_stage equal to 1, the target value for the stay_id will be 1.
# Otherwise, it will be 0.
print("choose one label for each stay_id (whenever it turn pos in the whole staying)")

def one_label_per_icustay(id_list, X):
    dataset = X
    temp_icustay_df = pd.DataFrame()
    target_list = []

    for icustay in id_list:
        temp_icustay_df = dataset.loc[dataset["stay_id"] == icustay].sort_values(by=["charttime"])
        if any(temp_icustay_df.aki_stage == 1):
            target_list.append(1)
        else:
            target_list.append(0)

    return target_list


target_list = one_label_per_icustay(id_list, X)

target = pd.DataFrame()
target["stay_id"] = id_list
target["y_true"] = target_list

choose one label for each stay_id (whenever it turn pos in the whole staying)


In [59]:
target

,stay_id,y_true
0,30000153,1
1,30000213,1
2,30000484,1
3,30000646,0
4,30001148,1
...,...,...
65408,39999286,1
65409,39999384,0
65410,39999552,0
65411,39999562,0


In [60]:
print("number of neg and pos label in target(whole stay)")
target["y_true"].value_counts()

number of neg and pos label in target(whole stay)


y_true
1    44654
0    20759
Name: count, dtype: int64

In [61]:
hour = 48  # set time span
print("calculate how many pos label within the first 48hrs, could be different time span")

def count_pos_label(X, INTIME, hour):
    dataset = X
    dataset = pd.merge(dataset, INTIME, on=["stay_id"], how="left", copy=False)
    dataset["HOURS"] = (dataset.charttime - dataset.intime).apply(lambda s: s / np.timedelta64(1, "s")) / 60.0 / 60
    dataset = dataset[dataset["HOURS"] >= 0]
    dataset = dataset[dataset["HOURS"] <= hour]
    dataset = dataset.reset_index(drop=True)

    temp_icustay_df = pd.DataFrame()
    target_list = []

    for icustay in id_list:
        temp_icustay_df = dataset.loc[dataset["stay_id"] == icustay].sort_values(by=["charttime"])
        if any(temp_icustay_df.aki_stage == 1):
            target_list.append(1)
        else:
            target_list.append(0)
    print("number of neg and pos label within the first " + str(hour) + "hr")
    print(Counter(target_list))


count_pos_label(X, INTIME, hour)

# TODO 3/24: also compute within 24 hours
hour = 24
count_pos_label(X, INTIME, hour)

calculate how many pos label within the first 48hrs, could be different time span
number of neg and pos label within the first 48hr
Counter({1: 39875, 0: 25538})
number of neg and pos label within the first 24hr
Counter({0: 33035, 1: 32378})


In [7]:
# Aqui va la cargada del archivo Y_copy_2
# Y = pd.read_csv("../data/Y_copy_2.csv", index_col=0)
# Y["charttime"] = pd.to_datetime(Y["charttime"])

# ⭐ Checkpoint #1

In [63]:
X_copy_extract_label = X.copy()

In [109]:
print(X_copy_extract_label.shape)
print(X_copy_extract_label.columns)
X_copy_extract_label.head()

(11163687, 31)
Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg', 'heartrate_mean', 'sysbp_mean', 'diasbp_mean',
       'resprate_mean', 'spo2_mean', 'vent', 'vasopressor', 'sedative'],
      dtype='object')


,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,...,wbc_avg,glucose_avg,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,vent,vasopressor,sedative
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,12466550.0,23998182.0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN
2,12466550.0,23998182.0,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN
3,12466550.0,23998182.0,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,100.0,136.0,74.0,NaN,NaN,NaN,NaN,NaN
4,12466550.0,23998182.0,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,18.0,NaN,NaN,NaN,NaN


In [112]:
import pickle

# Save the DataFrame to a pickle file
with open('X_copy_extract_label.pkl', 'wb') as file:
    pickle.dump(X_copy_extract_label, file)

# Save the INTIME DataFrame to a pickle file
with open('INTIME.pkl', 'wb') as file:
    pickle.dump(INTIME, file)

# save dataset_detail
with open('dataset_detail.pkl', 'wb') as file:
    pickle.dump(dataset_detail, file)

# save target
with open('target.pkl', 'wb') as file:
    pickle.dump(target, file)

In [8]:
import pickle

# Load the DataFrame from the pickle file
with open('X_copy_extract_label.pkl', 'rb') as file:
    X_copy_extract_label = pickle.load(file)
    X = X_copy_extract_label.copy()

# Load the INTIME DataFrame from the pickle file
with open('INTIME.pkl', 'rb') as file:
    INTIME = pickle.load(file)

# Load dataset_detail
with open('dataset_detail.pkl', 'rb') as file:
    dataset_detail = pickle.load(file)

# Load target
with open('target.pkl', 'rb') as file:
    target = pickle.load(file)

In [68]:
print(X_copy_extract_label.shape)
X_copy_extract_label.head()

(11163687, 31)


,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,...,wbc_avg,glucose_avg,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,vent,vasopressor,sedative
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,12466550.0,23998182.0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN
2,12466550.0,23998182.0,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN
3,12466550.0,23998182.0,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,100.0,136.0,74.0,NaN,NaN,NaN,NaN,NaN
4,12466550.0,23998182.0,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,18.0,NaN,NaN,NaN,NaN


In [70]:
dataset_detail.head()

,subject_id,hadm_id,stay_id,admission_age,icu_intime,icu_outtime,anchor_age,anchor_year,anchor_year_group,deathtime,...,WHITE - RUSSIAN,AMBULATORY OBSERVATION,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
18052,12466550,23998182,30000153,61.743196,2174-09-29 12:09:00,2174-10-01 03:26:10,61,2174,2008 - 2010,NaN,...,False,False,False,False,False,False,True,False,False,False
23204,13180007,27543152,30000213,66.470084,2162-06-21 05:38:00,2162-06-22 20:52:48,64,2160,2017 - 2019,NaN,...,False,False,False,False,False,False,True,False,False,False
61611,18421337,22413411,30000484,92.036911,2136-01-14 17:23:32,2136-01-17 04:53:08,91,2135,2008 - 2010,NaN,...,False,False,False,False,False,False,True,False,False,False
16191,12207593,22795209,30000646,44.319070,2194-04-29 01:39:22,2194-05-03 18:23:48,43,2193,2011 - 2013,2194-05-06 02:29:00,...,False,False,False,False,False,False,True,False,False,False
21765,12980335,23552849,30001148,68.661177,2156-08-30 11:10:59,2156-08-31 14:25:34,68,2156,2008 - 2010,NaN,...,False,False,False,False,False,False,False,False,False,True


In [71]:
# X_copy_extract_label, dataset_detail
merged_data = pd.merge(X_copy_extract_label, dataset_detail, on=["subject_id", "hadm_id", "stay_id"], how="left")

In [75]:
print(merged_data.shape)
merged_data.head()

(11163687, 90)


,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,...,WHITE - RUSSIAN,AMBULATORY OBSERVATION,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
1,12466550.0,23998182.0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
2,12466550.0,23998182.0,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
3,12466550.0,23998182.0,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
4,12466550.0,23998182.0,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False


In [74]:
merged_data.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg', 'heartrate_mean', 'sysbp_mean', 'diasbp_mean',
       'resprate_mean', 'spo2_mean', 'vent', 'vasopressor', 'sedative',
       'admission_age', 'icu_intime', 'icu_outtime', 'anchor_age',
       'anchor_year', 'anchor_year_group', 'deathtime', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime', 'F', 'M',
       'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN', 'ASIAN - ASIAN INDIAN',
       'ASIAN - CHINESE', 'ASIAN - KOREAN', 'ASIAN - SOUTH EAST ASIAN',
       'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN', 'BLACK/CAP

In [92]:
# Create new columns to indicate ethnicity groups
merged_data['HISPANIC_LATINO'] = merged_data[
    ['HISPANIC OR LATINO', 'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISPANIC/LATINO - COLUMBIAN', 
     'HISPANIC/LATINO - CUBAN', 'HISPANIC/LATINO - DOMINICAN', 'HISPANIC/LATINO - GUATEMALAN', 
     'HISPANIC/LATINO - HONDURAN', 'HISPANIC/LATINO - MEXICAN', 'HISPANIC/LATINO - PUERTO RICAN', 
     'HISPANIC/LATINO - SALVADORAN']
].any(axis=1)

merged_data['ASIAN'] = merged_data[
    ['ASIAN', 'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN', 'ASIAN - SOUTH EAST ASIAN']
].any(axis=1)

merged_data['BLACK_AFRICAN'] = merged_data[
    ['BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN', 'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND']
].any(axis=1)

merged_data['WHITE'] = merged_data[
    ['WHITE', 'WHITE - BRAZILIAN', 'WHITE - EASTERN EUROPEAN', 'WHITE - OTHER EUROPEAN', 'WHITE - RUSSIAN']
].any(axis=1)

# Native -> 'AMERICAN INDIAN/ALASKA NATIVE'
# 'UNKNOWN'

merged_data['OTHER'] = merged_data[
    ['MULTIPLE RACE/ETHNICITY', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 
     'OTHER', 'PATIENT DECLINED TO ANSWER', 'PORTUGUESE', 'SOUTH AMERICAN', 'UNABLE TO OBTAIN']
].any(axis=1)

# Convert boolean columns to integers
ethnicity_columns = ['HISPANIC_LATINO', 'ASIAN', 'BLACK_AFRICAN', 'WHITE', 'OTHER']
merged_data[ethnicity_columns] = merged_data[ethnicity_columns].astype(int)

# Display the updated dataframe
merged_data.head()

,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,...,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT,HISPANIC_LATINO,BLACK_AFRICAN
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,0,0
1,12466550.0,23998182.0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,0,0
2,12466550.0,23998182.0,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,0,0
3,12466550.0,23998182.0,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,0,0
4,12466550.0,23998182.0,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,0,0


## Generate statistics

In [139]:
print('glucose_avg')
print(merged_data['glucose_avg'].mean())
print(merged_data['glucose_avg'].std())
print(merged_data['glucose_avg'].isna().sum())

print('hematocrit_avg')
print(merged_data['hematocrit_avg'].mean(), merged_data['hematocrit_avg'].std(), merged_data['hematocrit_avg'].isna().sum())

print('hemoglobin_avg')
print(merged_data['hemoglobin_avg'].mean(), merged_data['hemoglobin_avg'].std(), merged_data['hemoglobin_avg'].isna().sum())

print('wbc_avg')
print(merged_data['wbc_avg'].mean(), merged_data['wbc_avg'].std(), merged_data['wbc_avg'].isna().sum())

glucose_avg
182.65281853620584
6289.787734579031
9195546
hematocrit_avg
29.29485896973133 5.566497140797459 10038893
hemoglobin_avg
9.61662518159898 1.9246151099576845 10120179
wbc_avg
11.273508011588346 9.238442995607494 10222051


In [141]:
print("Vital signs statistics")

print('diasbp_mean')
print(merged_data['diasbp_mean'].mean(), merged_data['diasbp_mean'].std(), merged_data['diasbp_mean'].isna().sum())

print('sysbp_mean')
print(merged_data['sysbp_mean'].mean(), merged_data['sysbp_mean'].std(), merged_data['sysbp_mean'].isna().sum())

print('heartrate_mean')
print(merged_data['heartrate_mean'].mean(), merged_data['heartrate_mean'].std(), merged_data['heartrate_mean'].isna().sum())

print('resprate_mean')
print(merged_data['resprate_mean'].mean(), merged_data['resprate_mean'].std(), merged_data['resprate_mean'].isna().sum())

print('spo2_mean')
print(merged_data['spo2_mean'].mean(), merged_data['spo2_mean'].std(), merged_data['spo2_mean'].isna().sum())

Vital signs statistics
diasbp_mean
62.77914666651626 15.215830065919057 5135399
sysbp_mean
119.64257572913658 22.618245078570382 5134021
heartrate_mean
86.34865423473872 18.30863189173154 4890431
resprate_mean
20.148651057222434 5.868035026105263 4890693
spo2_mean
96.77838046638628 3.266665324339711 5013120


In [ ]:
merged_data['creat'].value_counts()
merged_data['creat'].isna().sum()

5731355

In [111]:
# Convert 'icu_intime' and 'icu_outtime' to datetime if they are not already
merged_data['icu_intime'] = pd.to_datetime(merged_data['icu_intime'])
merged_data['icu_outtime'] = pd.to_datetime(merged_data['icu_outtime'])

# Calculate the length of stay in hours
merged_data['length_of_stay'] = (merged_data['icu_outtime'] - merged_data['icu_intime']).dt.total_seconds() / (3600 * 24)

print(merged_data['length_of_stay'].mean())
print(merged_data['length_of_stay'].std())

9.997556611848358
12.011719629909585


In [12]:
print("dataset drop AKI stages column")
X = X.drop(["aki_stage"], axis=1)

dataset drop AKI stages column


In [13]:
X = X.drop(["subject_id", "hadm_id"], axis=1)

In [10]:
X['subject_id'].nunique()

45233

In [15]:
print(X.shape)
X.columns

(11163687, 28)


Index(['stay_id', 'charttime', 'creat_low_past_7day', 'creat_low_past_48hr',
       'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr', 'aki_stage_crrt',
       'aki_stage_smoothed', 'creat', 'aniongap_avg', 'bicarbonate_avg',
       'chloride_avg', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'sodium_avg', 'bun_avg', 'wbc_avg', 'glucose_avg', 'heartrate_mean',
       'sysbp_mean', 'diasbp_mean', 'resprate_mean', 'spo2_mean', 'vent',
       'vasopressor', 'sedative'],
      dtype='object')

In [14]:
X.head()

,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage_smoothed,creat,...,wbc_avg,glucose_avg,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,vent,vasopressor,sedative
0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN
2,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN
3,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,100.0,136.0,74.0,NaN,NaN,NaN,NaN,NaN
4,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,18.0,NaN,NaN,NaN,NaN


# Resampling

In [ ]:
# label = ['aki_stage']
skip = ["stay_id", "charttime"]
if MAX_FEATURE_SET:
    discrete_feat = ["sedative", "vasopressor", "vent"]
    skip.extend(discrete_feat)
    # all features that are not in skip are numeric
    
numeric_feat = list(X.columns.difference(skip))

In [17]:
print("Discrete features: ", discrete_feat)
print("Numeric features: ", numeric_feat)

Discrete features:  ['sedative', 'vasopressor', 'vent']
Numeric features:  ['aki_stage_crrt', 'aki_stage_smoothed', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg', 'creat', 'creat_low_past_48hr', 'creat_low_past_7day', 'diasbp_mean', 'glucose_avg', 'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean', 'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg']


In [18]:
print("TIME_SAMPLING", TIME_SAMPLING)
print("MOST_COMMON",MOST_COMMON)
print("SAMPLING_INTERVAL",SAMPLING_INTERVAL)
print("MAX_FEATURE_SET", MAX_FEATURE_SET)

TIME_SAMPLING True
MOST_COMMON False
SAMPLING_INTERVAL 1H
MAX_FEATURE_SET True


In [ ]:
if TIME_SAMPLING and MOST_COMMON:
    print("resampling: MOST_COMMON with interval of " + str(SAMPLING_INTERVAL))
    # Resample the data using assigned interval,mode() for most common
    X = X.set_index("charttime").groupby("stay_id").resample(SAMPLING_INTERVAL).mode().reset_index()

elif TIME_SAMPLING:
    print("resampling: MEAN & ZERO with interval of " + str(SAMPLING_INTERVAL))
    # Sampling with different strategies per kind of variable
    # label = ['aki_stage']
    skip = ["stay_id", "charttime"] #'aki_stage'
    
    if MAX_FEATURE_SET:
        discrete_feat = ["sedative", "vasopressor", "vent"]
        # discrete_feat = ['sedative', 'vasopressor', 'vent', 'hadm_id'] # Like in original notebok
        skip.extend(discrete_feat)
    # all features that are not in skip are numeric
    numeric_feat = list(X.columns.difference(skip))

    # Applying aggregation to features depending on their type
    X = X.set_index("charttime").groupby("stay_id").resample(SAMPLING_INTERVAL)
    if MAX_FEATURE_SET:
        X_discrete = X[discrete_feat].max().fillna(FILL_VALUE).astype(np.int64)
    X_numeric = X[numeric_feat].mean()
    # X_label = X['aki_stage'].max()
    print("Merging sampled features")

    try:
        X = pd.concat([X_numeric, X_discrete], axis=1).reset_index() #X_label
    except:
        print("Exception")
        # X = pd.concat([X_numeric,X_label], axis=1).reset_index()
        X = X_numeric.reset_index()

print(X.shape)

# Label forward fill
# X['aki_stage'] = X['aki_stage'].ffill(limit=RESAMPLE_LIMIT)

resampling: MEAN & ZERO with interval of 1H
Merging sampled features
(18879678, 28)


In [46]:
#Exercise
data = {
    'charttime': pd.date_range(start='2021-01-01', periods=12, freq='H'),
    'stay_id': [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],
    'feature1': [1, 2, np.nan, 4, 5, np.nan, 7, 8, 9, np.nan, 11, 12],
    'feature2': [13, 14, 15, np.nan, 17, 18, np.nan, 20, 21, 22, np.nan, 24]
}
test = pd.DataFrame(data)
discrete_feat = ['feature1', 'feature2']

for name, group in test.set_index("charttime").groupby("stay_id"):
    print(f"Group: {name}")
    print(group)

Group: 1
                     stay_id  feature1  feature2
charttime                                       
2021-01-01 00:00:00        1       1.0      13.0
2021-01-01 01:00:00        1       2.0      14.0
2021-01-01 02:00:00        1       NaN      15.0
2021-01-01 03:00:00        1       4.0       NaN
Group: 2
                     stay_id  feature1  feature2
charttime                                       
2021-01-01 04:00:00        2       5.0      17.0
2021-01-01 05:00:00        2       NaN      18.0
2021-01-01 06:00:00        2       7.0       NaN
2021-01-01 07:00:00        2       8.0      20.0
2021-01-01 08:00:00        2       9.0      21.0
2021-01-01 09:00:00        2       NaN      22.0
2021-01-01 10:00:00        2      11.0       NaN
2021-01-01 11:00:00        2      12.0      24.0


In [43]:
result_ = test.set_index("charttime").groupby("stay_id").resample("3H")
print(result_.max())

                             stay_id  feature1  feature2
stay_id charttime                                       
1       2021-01-01 00:00:00        1       2.0      15.0
        2021-01-01 03:00:00        1       4.0       NaN
2       2021-01-01 03:00:00        2       5.0      18.0
        2021-01-01 06:00:00        2       9.0      21.0
        2021-01-01 09:00:00        2      12.0      24.0


In [74]:
X_copy_resampled = X.copy()

# fit time span with Yereva

In [45]:
X.shape

(18879678, 28)

In [77]:
INTIME.head()

,stay_id,intime
0,39553978,2180-07-23 14:00:00
1,39765666,2189-06-27 08:42:00
2,37067082,2157-11-20 19:18:02
3,34592300,2157-12-19 15:42:24
4,31205490,2110-04-11 15:52:22


In [47]:
print("Merging intime column to X")
print("Drop rows that has HOURS > 48h, could be other time span. And drop rows that has HOURS < 0")
X = pd.merge(X, INTIME, on=["stay_id"], how="left", copy=False)
X["HOURS"] = (X.charttime - X.intime).apply(lambda s: s / np.timedelta64(1, "s")) / 60.0 / 60
X = X[X["HOURS"] >= 0]
X = X[X["HOURS"] <= MAX_HOUR]
X = X.reset_index(drop=True)

Merging intime column to X
Drop rows that has HOURS > 48h, could be other time span. And drop rows that has HOURS < 0


In [48]:
X.shape

(3079482, 30)

# Imputing 

In [50]:
X.columns

Index(['stay_id', 'charttime', 'aki_stage_crrt', 'aki_stage_smoothed',
       'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg', 'creat',
       'creat_low_past_48hr', 'creat_low_past_7day', 'diasbp_mean',
       'glucose_avg', 'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg',
       'potassium_avg', 'resprate_mean', 'sodium_avg', 'spo2_mean',
       'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg',
       'sedative', 'vasopressor', 'vent', 'intime', 'HOURS'],
      dtype='object')

In [51]:
print("IMPUTE_EACH_ID", IMPUTE_EACH_ID)
print("IMPUTE_COLUMN", IMPUTE_COLUMN)

IMPUTE_EACH_ID True
IMPUTE_COLUMN False


In [52]:
print("Imputation.")
# X['aki_stage'] = X['aki_stage'].fillna(0)
remove_list = ["stay_id", "charttime", "intime", "HOURS"]

# using most common within each stay_id
if IMPUTE_EACH_ID:
    column_name = list(X.columns)
    for item in remove_list:
        column_name.remove(item)
    # column_name.remove(column_name[0]) 
    for feature in column_name:
        X.loc[X[feature].isnull(), feature] = X.stay_id.map(
            # Calculate the mode of the feature for each stay_id
            fast_mode(X, ["stay_id"], feature).set_index("stay_id")[feature]
        )

# imputation based on whole column
if IMPUTE_COLUMN:
    imp = SimpleImputer(missing_values=np.nan, strategy=IMPUTE_METHOD)
    cols = list(X.columns)
    for item in remove_list:
        cols.remove(item)
    X[cols] = imp.fit_transform(X[cols])

# If no imputation method selected or only impute each id, for the remaining nan impute direclty with FILL_VALUE
X = X.fillna(FILL_VALUE)

Imputation.


In [69]:
# temp = X.copy()

# X = temp.copy()

In [53]:
# more comfortable to review in this order
print("check variables")
try:
    cols = ['stay_id', 'charttime','aniongap_avg','bicarbonate_avg', 'bun_avg','chloride_avg',
            'creat','diasbp_mean', 'glucose_avg', 'heartrate_mean', 'hematocrit_avg','hemoglobin_avg', 
            'potassium_avg', 'resprate_mean', 'sodium_avg','spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 
            'uo_rt_24hr', 'uo_rt_6hr','wbc_avg', 'sedative', 'vasopressor', 'vent',"HOURS" , "intime"]
    X = X[cols]
    print("success")
except:
    try:
        cols = ['stay_id', 'charttime','aki_stage','creat','uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr']
        X = X[cols]
        print("try")
    except:
        print("error")

check variables
success


In [54]:
X.head()

,stay_id,charttime,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,...,sysbp_mean,uo_rt_12hr,uo_rt_24hr,uo_rt_6hr,wbc_avg,sedative,vasopressor,vent,HOURS,intime
0,30000153,2174-09-29 13:00:00,12.0,19.0,22.0,110.0,0.8,74.5,158.0,104.0,...,132.0,0.5321,0.5827,0.7045,11.9,1,0,1,0.85,2174-09-29 12:09:00
1,30000153,2174-09-29 14:00:00,12.0,19.0,22.0,112.0,0.8,61.0,176.0,83.0,...,131.0,0.5321,0.5827,0.7045,11.9,0,0,1,1.85,2174-09-29 12:09:00
2,30000153,2174-09-29 15:00:00,12.0,19.0,22.0,115.0,0.9,65.0,192.0,92.0,...,123.0,0.5321,0.5827,0.7045,17.0,1,0,1,2.85,2174-09-29 12:09:00
3,30000153,2174-09-29 16:00:00,12.0,19.0,22.0,115.0,0.8,55.0,175.0,83.0,...,109.0,0.5321,0.5827,0.7045,11.9,1,0,1,3.85,2174-09-29 12:09:00
4,30000153,2174-09-29 17:00:00,12.0,19.0,22.0,115.0,0.8,56.0,98.0,103.0,...,111.0,0.5321,0.5827,0.7045,11.9,0,0,1,4.85,2174-09-29 12:09:00


# Shifting labels

In [170]:
#print("Shifting the labels 48 h") # by 8 position : 6h sampling*8=48h and ffil 8 newly empty ones
# X['aki_stage'] = X.groupby('stay_id')['aki_stage'].shift(-(HOURS_AHEAD // int(SAMPLING_INTERVAL[:-1])))
# X = X.dropna(subset=['aki_stage'])
# X['stay_id'].nunique()

65364

# Add categorical features (details)

In [56]:
dataset_detail.head()

,subject_id,hadm_id,stay_id,admission_age,icu_intime,icu_outtime,anchor_age,anchor_year,anchor_year_group,deathtime,...,WHITE - RUSSIAN,AMBULATORY OBSERVATION,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
18052,12466550,23998182,30000153,61.743196,2174-09-29 12:09:00,2174-10-01 03:26:10,61,2174,2008 - 2010,NaN,...,False,False,False,False,False,False,True,False,False,False
23204,13180007,27543152,30000213,66.470084,2162-06-21 05:38:00,2162-06-22 20:52:48,64,2160,2017 - 2019,NaN,...,False,False,False,False,False,False,True,False,False,False
61611,18421337,22413411,30000484,92.036911,2136-01-14 17:23:32,2136-01-17 04:53:08,91,2135,2008 - 2010,NaN,...,False,False,False,False,False,False,True,False,False,False
16191,12207593,22795209,30000646,44.319070,2194-04-29 01:39:22,2194-05-03 18:23:48,43,2193,2011 - 2013,2194-05-06 02:29:00,...,False,False,False,False,False,False,True,False,False,False
21765,12980335,23552849,30001148,68.661177,2156-08-30 11:10:59,2156-08-31 14:25:34,68,2156,2008 - 2010,NaN,...,False,False,False,False,False,False,False,False,False,True


In [86]:
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime', 'intime'],
      dtype='object')

In [85]:
# dataset_detail[['admission_age', 'anchor_age']].tail()
dataset_detail[['icu_intime','intime']].head()

,icu_intime,intime
0,2180-07-23 14:00:00,2180-07-23 14:00:00
1,2189-06-27 08:42:00,2189-06-27 08:42:00
2,2157-11-20 19:18:02,2157-11-20 19:18:02
3,2157-12-19 15:42:24,2157-12-19 15:42:24
4,2110-04-11 15:52:22,2110-04-11 15:52:22


In [87]:
dataset_detail_copy = dataset_detail.copy()

In [ ]:
if MAX_FEATURE_SET:
    # extract datasets based on id_list
    dataset_detail = dataset_detail.loc[dataset_detail["stay_id"].isin(id_list)]
    # sort by ascending order
    dataset_detail = dataset_detail.sort_values(by=["stay_id"])
    # subject_id = dataset_detail["subject_id"].unique()
    # print(dataset_detail)

    # transfrom categorical data to binary form
    dataset_detail = dataset_detail.drop(["intime"], axis=1)
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("gender")))
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("race")))
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("admission_type")))
    # X = X.drop(['subject_id', 'hadm_id'], axis=1)
    # dataset_detail = dataset_detail.drop(['subject_id', 'hadm_id'], axis=1)
    X = pd.merge(X, dataset_detail, on=["stay_id"], how="left", copy=False)
    numeric_feat.append("anchor_age")

In [ ]:
# If checkpoint 1 is loaded, the following cell should be executed to load the dataset_detail DataFrame.
X = pd.merge(X, dataset_detail, on=["stay_id"], how="left", copy=False)
numeric_feat.append("anchor_age")

In [58]:
X.head()

,stay_id,charttime,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,...,WHITE - RUSSIAN,AMBULATORY OBSERVATION,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
0,30000153,2174-09-29 13:00:00,12.0,19.0,22.0,110.0,0.8,74.5,158.0,104.0,...,False,False,False,False,False,False,True,False,False,False
1,30000153,2174-09-29 14:00:00,12.0,19.0,22.0,112.0,0.8,61.0,176.0,83.0,...,False,False,False,False,False,False,True,False,False,False
2,30000153,2174-09-29 15:00:00,12.0,19.0,22.0,115.0,0.9,65.0,192.0,92.0,...,False,False,False,False,False,False,True,False,False,False
3,30000153,2174-09-29 16:00:00,12.0,19.0,22.0,115.0,0.8,55.0,175.0,83.0,...,False,False,False,False,False,False,True,False,False,False
4,30000153,2174-09-29 17:00:00,12.0,19.0,22.0,115.0,0.8,56.0,98.0,103.0,...,False,False,False,False,False,False,True,False,False,False


In [59]:
X = X.drop(['charttime', 'intime'], axis=1)

In [60]:
feature_names = [
    "Anion gap",
    "Bicarbonate",
    "Blood Urea Nitrogen",
    "Chloride",
    "Creatinine",
    "Diastolic BP",
    "Glucose",
    "Heart rate",
    "Hematocrit",
    "Hemoglobin",
    "Potassium",
    "Respiratory rate",
    "Sodium",
    "Oxygen saturation",
    "Systolic BP",
    "Urine output 12h",
    "Urine output 24h",
    "Urine output 6h",
    "White cell count",
    "Sedative",
    "Vasopressor",
    "Ventilation",
    "Age",
    "Female gender",
    "Male gender",
    "Asian ethnicity",
    "Black ethnicity",
    "Hispanic ethnicity",
    "Native american",
    "Other ethnicity",
    "Ethnicity unknown",
    "White ethnicity",
    "Elective admission",
    "Emergency admission",
    "Urgent admission",
]
print(len(feature_names))

35


# Cap features between 0.01 / 0.99 quantile and normalisation

In [61]:
numeric_feat

['aki_stage_crrt',
 'aki_stage_smoothed',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'creat_low_past_48hr',
 'creat_low_past_7day',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'admission_age']

In [62]:
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'hadm_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISP

In [63]:
X = cap_data(X)

Capping between the 0.01 and 0.99 quantile
cap_maks Index(['AMBULATORY OBSERVATION', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'DIRECT EMER.',
       'DIRECT OBSERVATION', 'ELECTIVE', 'EU OBSERVATION', 'EW EMER.', 'F',
       'HISPANIC OR LATINO', 'HISPANIC/LATINO - CENTRAL AMERICAN',
       'HISPANIC/LATINO - COLUMBIAN', 'HISPANIC/LATINO - CUBAN',
       'HISPANIC/LATINO - DOMINICAN', 'HISPANIC/LATINO - GUATEMALAN',
       'HISPANIC/LATINO - HONDURAN', 'HISPANIC/LATINO - MEXICAN',
       'HISPANIC/LATINO - PUERTO RICAN', 'HISPANIC/LATINO - SALVADORAN',
       'HOURS', 'M', 'MULTIPLE RACE/ETHNICITY',
       'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 'OBSERVATION ADMIT',
       'OTHER', 'PATIENT DECLINED TO ANSWER', 'PORTUGUESE', 'SOUTH AMERICAN',
       'SURGICAL SAME DAY ADMISSION', 'U

In [64]:
features_to_remove = ['aki_stage_crrt', 'aki_stage_smoothed', 'creat_low_past_48hr', 'creat_low_past_7day']
numeric_feat = [feature for feature in numeric_feat if feature not in features_to_remove]
numeric_feat

['aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'admission_age']

In [96]:
X = normalise_data(X, numeric_feat)

Normalizing in [0,1] with min-max normalization


In [97]:
# X.loc[X["stay_id"] == 272725].sort_values(by=["HOURS"])["HOURS"]
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'hadm_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISP

In [65]:
# X = X.sort_values(by=['stay_id', 'charttime'])
seq_lengths = X.groupby(["stay_id"], as_index=False).size().sort_values(by=["size"], ascending=False)
sequence_length = seq_lengths.max()  # the longest sequence per icustay-id
print(sequence_length)

stay_id    39899718
size          30836
dtype: int64


In [66]:
# AL re-write as try except to make it work as hadm_id is not used if only one csv file is used and none are merged
try:
    X.drop(["hadm_id"], axis=1, inplace=True)
except:
    pass

In [67]:
X = X.sort_values(by=["subject_id", "HOURS"])
# features = X.shape[1]-3
# features

# Count number of variables for final dataset

In [101]:
feature_names

['Anion gap',
 'Bicarbonate',
 'Blood Urea Nitrogen',
 'Chloride',
 'Creatinine',
 'Diastolic BP',
 'Glucose',
 'Heart rate',
 'Hematocrit',
 'Hemoglobin',
 'Potassium',
 'Respiratory rate',
 'Sodium',
 'Oxygen saturation',
 'Systolic BP',
 'Urine output 12h',
 'Urine output 24h',
 'Urine output 6h',
 'White cell count',
 'Sedative',
 'Vasopressor',
 'Ventilation',
 'Age',
 'Female gender',
 'Male gender',
 'Asian ethnicity',
 'Black ethnicity',
 'Hispanic ethnicity',
 'Native american',
 'Other ethnicity',
 'Ethnicity unknown',
 'White ethnicity',
 'Elective admission',
 'Emergency admission',
 'Urgent admission']

In [102]:
print(len(X.columns))
X.columns

84


Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISPANIC/LATINO

In [103]:
features_to_extract = ['stay_id','charttime','aki_stage', 'aniongap_avg', 'bicarbonate_avg',
       'bun_avg', 'chloride_avg', 'creat', 'diasbp_mean', 'glucose_avg',
       'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'resprate_mean', 'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr',
       'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent',
       'anchor_age','F', 'M', 'ASIAN', 'BLACK/AFRICAN', 'HISPANIC OR LATINO', 'AMERICAN INDIAN/ALASKA NATIVE',
       'OTHER', 'UNKNOWN', 'WHITE', 'ELECTIVE', 'URGENT']

In [104]:
X_selected = X[features_to_extract]
X_selected

KeyError: "['charttime', 'aki_stage'] not in index"

In [105]:
features_list = list(X.columns)

# list of variables to be removed at the end
remove_list_final = ["stay_id", "subject_id"] # exluded: "F"
for item in remove_list_final:
    features_list.remove(item)

features = len(features_list)
print("number of features: " + str(features))

number of features: 82


# Split dataset

In [97]:
# len(id_list)
# id_list

In [98]:
# print("divide dataset into train, test and validation sets")
# id_train, id_test_val = train_test_split(id_list, test_size = SPLIT_SIZE, random_state = RANDOM_SEED) # train set is 80%)
# print("train is %d" % len(id_train))

# # remaining 20% split in halves as test and validation 10% and 10%
# id_valid, id_test = train_test_split(id_test_val, test_size = 0.5, random_state = RANDOM_SEED) # test 10% valid 10%
# print("val and test are %d" %len(id_test))

In [99]:
# train = X[X.stay_id.isin(id_train)].sort_values(by=['stay_id'])
# test = X[X.stay_id.isin(id_test)].sort_values(by=['stay_id'], ignore_index = True) 
# validation = X[X.stay_id.isin(id_valid)].sort_values(by=['stay_id']) 

# test = test.sort_values(by=['stay_id', 'charttime'], ignore_index = True)
# train = train.sort_values(by=['stay_id', 'charttime'], ignore_index = True)
# validation = validation.sort_values(by=['stay_id', 'charttime'], ignore_index = True)

# Random split subject ID into train(val), and test

In [100]:
if RANDOM_SPLIT:
    print("extract subject_id list")
    subject_id = X["subject_id"].unique()
    subject_id = np.sort(subject_id)

extract subject_id list


In [101]:
if RANDOM_SPLIT:
    print("number of unique subject id: " + str(len(subject_id)))

number of unique subject id: 44335


In [102]:
if RANDOM_SPLIT:
    print("RANDOM SPLIT")
    print("divide dataset into train, test and validation sets")
    id_train_val, id_test = train_test_split(subject_id, test_size=0.1, random_state=RANDOM_SEED)  # train set is 80%)
    print("test is %d" % len(id_test))
    # remaining 20% split in halves as test and validation 10% and 10%
    id_train, id_val = train_test_split(id_train_val, test_size=0.111, random_state=RANDOM_SEED)  # test 10% valid 10%
    print("train is %d" % len(id_train))
    print("val is %d" % len(id_val))

    # sort list
    id_test.sort()
    id_train.sort()
    id_val.sort()

RANDOM SPLIT
divide dataset into train, test and validation sets
test is 4434
train is 35471
val is 4430


# Use fixed id_list from Yereva

In [122]:
if FIXED:
    # the Yereva id files pathes are here
    DATA_PATH_yereva_test = "data/id_list_yereva/test_listfile.csv"
    DATA_PATH_yereva_train = "data/id_list_yereva/train_listfile.csv"
    DATA_PATH_yereva_val = "data/id_list_yereva/val_listfile.csv"

    print("read csv files")
    # reading csv files
    yereva_test = pd.read_csv(DATA_PATH_yereva_test, sep=",")
    yereva_train = pd.read_csv(DATA_PATH_yereva_train, sep=",")
    yereva_val = pd.read_csv(DATA_PATH_yereva_val, sep=",")

    # convert to list
    yereva_test = yereva_test["notes"].tolist()
    yereva_train = yereva_train["notes"].tolist()
    yereva_val = yereva_val["notes"].tolist()

    yereva_test_subject = []
    yereva_train_subject = []
    yereva_val_subject = []

    for subject in yereva_test:
        yereva_test_subject.append(int(subject.split("_")[0]))
    for subject in yereva_train:
        yereva_train_subject.append(int(subject.split("_")[0]))
    for subject in yereva_val:
        yereva_val_subject.append(int(subject.split("_")[0]))

In [123]:
if FIXED:
    id_test = []
    id_train = []
    id_val = []

    n = 0

    while n < len(subject_id):
        if subject_id[n] in yereva_test_subject:
            id_test.append(subject_id[n])
            n = n + 1
        elif subject_id[n] in yereva_train_subject:
            id_train.append(subject_id[n])
            n = n + 1
        elif subject_id[n] in yereva_val_subject:
            id_val.append(subject_id[n])
            n = n + 1
        else:
            n = n + 1

    id_test.sort()
    id_train.sort()
    id_val.sort()

    print("Fixed list from Yereva")
    print("test is %d" % len(id_test))
    print("train is %d" % len(id_train))
    print("val is %d" % len(id_val))

In [124]:
if FIXED:
    print("combine subject_id list")
    subject_id = id_test + id_train
    subject_id = subject_id + id_val
    subject_id.sort()

    print("number of unique subject id: " + str(len(subject_id)))

# Convert icustay data into individual timeseries csv

In [125]:
# str(list(target.loc[target['icustay_id'] == 237693]['y_true'])[0])

In [105]:
def convert_icustay_to_AKIfolder(dataset, subject_id, output_path, id_train, id_test, id_val, target):

    temp_icustay_list = []  # to store the stay_id under same subject_id
    n = 0  # index to loop through temp_icustay_liremove_list_finalst
    num_stay = 0
    dataset = X
    temp_dataset = pd.DataFrame()
    sub_temp_dataset = pd.DataFrame()

    train_pairs = []
    test_pairs = []
    val_pairs = []

    for subject in subject_id:
        # make path for subject folder
        dn = os.path.join(OUTPUT_PATH, str(subject))
        try:
            os.makedirs(dn)
        except:
            pass

        temp_dataset = dataset.loc[dataset["subject_id"] == subject].sort_values(by=["stay_id"])
        temp_icustay_list = temp_dataset["stay_id"].unique()
        num_stay = len(temp_icustay_list)
        print(num_stay)
        n = 0

        while n < num_stay:
            sys.stdout.write(
                "\rSUBJECT_ID: {0} STAY_ID: {1} Episode {2}...".format(subject, temp_icustay_list[n], n + 1)
            )
            sub_temp_dataset = temp_dataset.loc[temp_dataset["stay_id"] == temp_icustay_list[n]]
            sub_temp_dataset = sub_temp_dataset.drop(remove_list_final, axis=1)
            sub_temp_dataset = sub_temp_dataset.set_index('HOURS').sort_index(axis=0)

            sub_temp_dataset.to_csv(
                os.path.join(
                    OUTPUT_PATH,
                    str(subject),
                    "{}_episode{}_timeseries_{}.csv".format(subject, n + 1, temp_icustay_list[n]),
                ),
                index_label="Hours",
            )

            # create list for id list for train/test/val
            if subject in id_train:
                train_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )
            elif subject in id_test:
                test_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )
            elif subject in id_val:
                val_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )

            n = n + 1

    sys.stdout.write("DONE!\n")

    return train_pairs, test_pairs, val_pairs

In [106]:
train_pairs, test_pairs, val_pairs = convert_icustay_to_AKIfolder(
    X, subject_id, OUTPUT_PATH, id_train, id_test, id_val, target
)

650
SUBJECT_ID: 10111112 STAY_ID: 39899718 Episode 650...1
SUBJECT_ID: 10111636 STAY_ID: 31318658 Episode 1...3
SUBJECT_ID: 10112163 STAY_ID: 39668021 Episode 3...1
SUBJECT_ID: 10112484 STAY_ID: 38897817 Episode 1...1
SUBJECT_ID: 10112789 STAY_ID: 32483147 Episode 1...1
SUBJECT_ID: 10112984 STAY_ID: 31093528 Episode 1...1
SUBJECT_ID: 10113381 STAY_ID: 36312173 Episode 1...1
SUBJECT_ID: 10113512 STAY_ID: 30119329 Episode 1...1
SUBJECT_ID: 10113636 STAY_ID: 30643549 Episode 1...2
SUBJECT_ID: 10113751 STAY_ID: 35339370 Episode 2...1
SUBJECT_ID: 10113898 STAY_ID: 32000672 Episode 1...1
SUBJECT_ID: 10114694 STAY_ID: 35032951 Episode 1...1
SUBJECT_ID: 10114841 STAY_ID: 35255528 Episode 1...1
SUBJECT_ID: 10114932 STAY_ID: 39678298 Episode 1...1
SUBJECT_ID: 10115024 STAY_ID: 39765878 Episode 1...1
SUBJECT_ID: 10115044 STAY_ID: 36324659 Episode 1...2
SUBJECT_ID: 10115397 STAY_ID: 36639474 Episode 2...1
SUBJECT_ID: 10115812 STAY_ID: 37028700 Episode 1...1
SUBJECT_ID: 10116621 STAY_ID: 30354498 E

In [107]:
OUTPUT_PATH

'data/AKI_fts2'

# Move subject_timeseries.csv file to train/test folder 

In [108]:
def move_to_partition(subjects_root_path, patients, partition):
    if not os.path.exists(os.path.join(subjects_root_path, partition)):
        print(f"Creating  partition", os.path.join(subjects_root_path, partition))
        os.mkdir(os.path.join(subjects_root_path, partition))
    for patient in patients:
        src = os.path.join(subjects_root_path, str(patient))
        dest = os.path.join(subjects_root_path, partition)
        for filename in os.listdir(src):
            shutil.move(os.path.join(src, str(filename)), dest)
        os.rmdir(src)

In [109]:
move_to_partition(OUTPUT_PATH, id_train, "train")
move_to_partition(OUTPUT_PATH, id_val, "train")
move_to_partition(OUTPUT_PATH, id_test, "test")

Creating  partition data/AKI_fts2\train
Creating  partition data/AKI_fts2\test


# Create test_listfile.csv, train_listfile.csv, val_listfile.csv, and move to AKI folder

In [110]:
with open(os.path.join(OUTPUT_PATH, "train_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in train_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))
        
with open(os.path.join(OUTPUT_PATH, "val_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in val_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))

with open(os.path.join(OUTPUT_PATH, "test_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in test_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))